## **Silver Notebook - Reconciliation Logic**
batch is authorative when present, stream fills in the gaps for orders too recent for the batch <br>
PS: attach a data source i.e. OneLake's LH for the script to effectively communicate with

In [3]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# constants
LAKEHOUSE = 'Lakehouse_1'

StatementMeta(, 87b02346-4953-49ed-b0d6-fe65af6349b2, 5, Finished, Available, Finished, False)

In [1]:
bronze_batch = spark.read.table("bronze.bronze_orders_batch")
bronze_stream = spark.read.table("bronze.bronze_orders_stream")

print(f"bronze batch rows = {bronze_batch.count()}")
print(f"bronze stream rows = {bronze_stream.count()}")

StatementMeta(, 87b02346-4953-49ed-b0d6-fe65af6349b2, 3, Finished, Available, Finished, False)

bronze batch rows = 419
bronze stream rows = 755


collapse stream events to latest state per order<br>
one order can emit multiple events ( Created, Updated, Cancelled) so keep only the most recent event per order_id

In [4]:
stream_window = Window.partitionBy("order_id").orderBy(F.col("event_ts").desc())
 
stream_latest = (
    bronze_stream
    .withColumn("rn", F.row_number().over(stream_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn(
        "stream_derived_status",
        F.when(F.col("event_type") == "OrderCreated", "Placed")
         .when(F.col("event_type") == "OrderUpdated", "Updated")
         .when(F.col("event_type") == "OrderCancelled", "Cancelled")
         .otherwise("Unknown")
    )
    .select(
        F.col("order_id"),
        F.col("customer_id").alias("stream_customer_id"),
        F.col("product_id").alias("stream_product_id"),
        F.col("quantity").alias("stream_quantity"),
        F.col("unit_price").alias("stream_unit_price"),
        F.col("event_ts").alias("stream_last_event_ts"),
        F.col("stream_derived_status"),
    )
)
 
print(f"Distinct orders seen in stream: {stream_latest.count()}")

StatementMeta(, 87b02346-4953-49ed-b0d6-fe65af6349b2, 6, Finished, Available, Finished, False)

Distinct orders seen in stream: 500


collapse batch to latest extract per order<br>
bronze_orders_batch is append-only across days, so an order placed a few days ago may have multiple extract_date rows e.g. Shipped -> Delivered). Keep the most recent extract.


In [5]:
batch_window = Window.partitionBy("order_id").orderBy(F.col("extract_date").desc())
 
batch_latest = (
    bronze_batch
    .withColumn("rn", F.row_number().over(batch_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
 
print(f"Distinct orders in latest batch extract: {batch_latest.count()}")

StatementMeta(, 87b02346-4953-49ed-b0d6-fe65af6349b2, 7, Finished, Available, Finished, False)

Distinct orders in latest batch extract: 419


**<mark>Reconciliation Logic</mark>**: full outer join, batch wins where present

In [6]:
reconciled = (
    batch_latest.alias("b")
    .join(stream_latest.alias("s"), on="order_id", how="full_outer")
    .select(
        F.col("order_id"),
        F.coalesce(F.col("b.customer_id"), F.col("s.stream_customer_id")).alias("customer_id"),
        F.coalesce(F.col("b.product_id"), F.col("s.stream_product_id")).alias("product_id"),
        F.coalesce(F.col("b.quantity"), F.col("s.stream_quantity")).alias("quantity"),
        F.coalesce(F.col("b.unit_price"), F.col("s.stream_unit_price")).alias("unit_price"),
        F.coalesce(F.col("b.discount_pct"), F.lit(0.0)).alias("discount_pct"),
        F.coalesce(F.col("b.order_ts"), F.col("s.stream_last_event_ts")).alias("order_ts"),
        # batch status wins when present; otherwise fall back to stream-derived status
        F.coalesce(F.col("b.status"), F.col("s.stream_derived_status")).alias("status"),
        F.col("b.payment_status").alias("payment_status"),          # null until batch confirms
        F.col("b.fulfillment_center").alias("fulfillment_center"),  # null until batch confirms
        F.when(F.col("b.order_id").isNotNull(), F.lit("ERP_BATCH"))
         .otherwise(F.lit("WEB_STREAM")).alias("source_system"),
        F.when(F.col("b.order_id").isNotNull() & F.col("s.order_id").isNotNull(), F.lit(True))
         .otherwise(F.lit(False)).alias("is_reconciled"),
        F.current_timestamp().alias("silver_updated_at"),
    )
)
 
print(f"Reconciled row count: {reconciled.count()}")
display(reconciled.limit(20))

StatementMeta(, 87b02346-4953-49ed-b0d6-fe65af6349b2, 8, Finished, Available, Finished, False)

Reconciled row count: 500


SynapseWidget(Synapse.DataFrame, fb7ac152-a74b-4768-b510-b8a93bcb3575)

MERGE into silver_orders (create table on first run)

In [7]:
if not spark.catalog.tableExists("silver_orders"):
    reconciled.write.format("delta").saveAsTable("silver_orders")
    print("Created silver_orders (first run).")
else:
    silver_table = DeltaTable.forName(spark, "silver_orders")
    (
        silver_table.alias("target")
        .merge(
            reconciled.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Merged into existing silver_orders.")

StatementMeta(, 87b02346-4953-49ed-b0d6-fe65af6349b2, 9, Finished, Available, Finished, False)

Created silver_orders (first run).


In [8]:
result = spark.read.table("silver_orders")
print(f"silver_orders total rows: {result.count()}")
result.groupBy("source_system", "is_reconciled").count().show()

StatementMeta(, 87b02346-4953-49ed-b0d6-fe65af6349b2, 10, Finished, Available, Finished, False)

silver_orders total rows: 500
+-------------+-------------+-----+
|source_system|is_reconciled|count|
+-------------+-------------+-----+
|    ERP_BATCH|         true|  419|
|   WEB_STREAM|        false|   81|
+-------------+-------------+-----+



In [9]:
%%sql
select * from silver_orders limit 20;

StatementMeta(, 87b02346-4953-49ed-b0d6-fe65af6349b2, 11, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 13 fields>